# Fair and Explainable Student Dropout Risk Prediction for Early Academic Support
### Technical presentation · Student Success Navigator · v1.0.0

**Question.** At the end of a student's first semester, can we rank students so that a small, voluntary adviser outreach list reaches those most likely to leave, using only information available at that point, explainably and with a fairness audit?

**Answer in one line.** A calibrated random-forest pipeline on 21 enrollment-time and first-semester features reaches held-out PR-AUC 0.817 (base rate 0.321), fills an illustrative outreach list at 96% precision, and shows an equal-opportunity gap by age that advisers must compensate for.

*Every number is read from files under `reports/`; nothing on these slides is typed by hand. Slides built from `notebooks/90_technical_deck.ipynb`.*

## 1 · Problem framing

- **Unit of analysis:** one student enrollment record · **Prediction point:** end of first semester
- **Task:** binary classification, `is_dropout` = 1 where source Target = Dropout (Enrolled and Graduate = 0)
- **Use:** decision support for *voluntary, supportive* adviser outreach within a fixed capacity
- **Non-use:** no automated or adverse decision about admission, enrollment, aid, grades, discipline, housing
- **Primary metric:** PR-AUC (imbalance: positive rate 0.321) · **Intervention metric:** Recall@K and Precision@K
- **Capacity assumption (ILLUSTRATIVE):** 10 students/week × 5 weeks = K 50 per cohort of 885
- Governance: constitution with 12 principles and 10 quality gates; Spec Kit spec → plan → 92 tasks

## 2 · Dataset

- UCI ML Repository 697, *Predict Students' Dropout and Academic Success* (Realinho et al., 2021), **CC BY 4.0**, DOI 10.24432/C5MC89
- One Portuguese higher-education institution; **4,424** records × 36 features + Target; 0 missing, 0 duplicates, 0 undocumented codes
- Target: Graduate 2,209 · Dropout 1,421 · Enrolled 794 → positive rate 0.321
- Every categorical encoding copied from the UCI variables table and verified against observed codes (22 coded columns, including Gender 1 = male, 0 = female)
- Stratified 80/20 split, seed 42: train **3539** · test **885** (evaluated once)
- Not representative of Philippine or other institutions; outcomes are historical and may encode institutional inequities

## 3 · Leakage controls (the feature-availability list)

| Class | Columns | Model input? |
|---|---|---|
| Enrollment-time features | 15 | yes |
| First-semester features | 6 | yes |
| Second-semester (after prediction point) | 6 | **no** |
| Financial status, undocumented timing (Debtor, Tuition fees up to date, Scholarship holder) | 3 | **no** (ambiguous) |
| Sensitive attributes (gender, age, nationality, international, marital, special needs) | 6 | **no** (audit only) |

- Enforced by `configs/features.yaml`, `project_features` / `assert_frame_allowed`, and tests that spy on every parquet read
- **Cost of the guard, measured:** including the 3 ambiguous columns would add 0.036–0.039 CV PR-AUC (`ablation_ambiguous.csv`) — a gain that may be leakage, so they stay out
- Excluding the 6 sensitive attributes costs at most 0.0065 PR-AUC (`ablation_sensitive.csv`)

## 4 · EDA (training split only)

<img src="../reports/figures/eda_correlation_spearman.png" width="620"/>

- First-semester approvals, approval rate and grade carry the strongest associations with dropout
- 138 training records have zero first-semester units enrolled (rate features undefined → imputed inside CV)
- Sensitive-group base rates differ widely (e.g. dropout rate 46.1% male vs 24.5% female on test): context for the audit, never a model input

## 5 · Features, selection, PCA

- **7 stateless engineered features** from allow-listed inputs: approval rate, evaluation participation rate, non-evaluation rate, grade change vs admission (scales matched), credited share, load, any-approved
- Preprocessing = median impute · scale · one-hot → **207** columns; all fitting inside CV folds
- **Selection:** mutual-information filter and L1-logistic embedded selectors over k ∈ {10…all}; keeping all 207 scored highest (0.818), embedded k = 30 within one std (0.811) → carried as a tuned variant, not imposed
- **PCA:** 42 components for 95% variance; PCA pipeline 0.801 vs 0.818 without → retained for analysis/visualisation only

<img src="../reports/figures/pca_scree.png" width="520"/>

## 6 · Model comparison (5-fold stratified CV, seed 42)

| Model (class-weighted, all features) | PR-AUC | ROC-AUC | Precision@K | Brier | ECE |
|---|---|---|---|---|---|
| Dummy (prior) | 0.321 | 0.500 | 0.33 | 0.218 | 0.001 |
| Logistic regression | 0.818 ± 0.017 | 0.878 | 0.97 | 0.134 | 0.096 |
| Random forest | 0.807 ± 0.009 | 0.872 | 0.96 | 0.126 | 0.032 |
| HistGradientBoosting | 0.811 ± 0.014 | 0.869 | 0.98 | 0.132 | 0.067 |

- SMOTENC lowered PR-AUC for both models it was tried on → class weighting kept
- After tuning (RandomizedSearchCV, PR-AUC), all three remain within one CV std → selection by documented rule: Recall@K, then **calibration error**, then fairness gap, then Brier, then fit time. **Chosen: random forest, all features**; isotonic calibration applied (ECE 0.054 → 0.017)
- Accuracy is reported but excluded from selection

## 7 · Final held-out evaluation (once; `test_evaluations` = 1)

| | PR-AUC | ROC-AUC | Recall@K | Precision@K | Brier | ECE |
|---|---|---|---|---|---|---|
| **Final calibrated RF** | **0.817** | 0.892 | 0.169 | 0.96 | 0.118 | 0.034 |
| Dummy baseline | 0.321 | 0.500 | 0.074 | 0.42 | 0.218 | 0.000 |

- Threshold 0.9728 = OOF score at the capacity selection rate (200 of 3539); on test it selects 57 students at precision 0.965
- **Recall@K is capacity-bound:** K = 50 slots for 284 eventual dropouts → ceiling ≈ 0.176; 10-week window reaches 0.327
- Test PR-AUC matches the CV estimate (0.808): no sign of selection overfitting

<img src="../reports/figures/test_pr_curve.png" width="480"/>

## 8 · Explainability

<img src="../reports/explainability/shap_global_bar.png" width="640"/>

- SHAP TreeExplainer on each of the 5 calibrated ensemble members, aggregated to source features, averaged; explains uncalibrated contributions
- Top drivers: sem1_approval_rate, Curricular units 1st sem (approved), Curricular units 1st sem (grade), grade_diff_vs_admission, Application mode
- Local TP / FP / FN / TN cases rendered into supportive adviser phrases (`configs/language.yaml`); sensitive attributes can never appear as reasons (tested)
- PDP/ICE for 9 continuous raw features; engineered features explained via SHAP only

## 9 · Fairness audit (aggregate, held-out cohort, verified encodings)

| Attribute | Selection-rate gap (DP) | DI ratio | Equal-opportunity gap (TPR) |
|---|---|---|---|
| Gender (female n=575, male n=310) | 0.040 | 0.56 | 0.010 |
| Age band (4 bands, smallest n=85) | 0.227 | 0.08 | **0.397** |

- Gender: list composition mirrors a two-fold base-rate difference; TPR nearly equal (0.189 vs 0.199); FPR < 1% both
- **Age: the material finding.** TPR 0.10 for 17–19 vs 0.50 for 35+: younger dropouts are under-reached through proxies (over-23 application route, programme, schedule)
- Mitigations on training OOF: reweighting (deployable; small parity gain, worse EO and calibration → not adopted); group thresholds (closes gender gap, needs gender at decision time → reported, not deployed)
- Recommended: adviser-side outreach-list allocation for younger bands; Equity Dashboard shows the gap each cycle

## 10 · Reproducibility and engineering

- Single seed (42), config-driven CLI (`python -m ssn …`), pinned `requirements.txt`, `n_jobs=1` final fit
- Persisted pipeline + manifest: git SHA, config hash, pipeline SHA-256, library versions, exact 21-column input schema, threshold, bands, test counter
- **Fresh-environment re-run: maximum absolute delta 0** across every compared table (`docs/REPRODUCIBILITY.md`)
- 165 automated tests: leakage (allow-list, fit isolation, parquet-read spies), privacy (no labels/sensitive columns in the demo cohort), metrics (hand-computed), artifact loading, language rules; CI on every push (lint, secrets scan, tests, language scan)

## 11 · Limitations and what this does not establish

- **Scope:** one institution's history; no transfer to other institutions without re-training and re-auditing
- **Proxies:** parental qualification/occupation, application route, programme, schedule re-encode age and class
- **Leakage guard cost:** 3 excluded financial columns worth ≈ 0.037 PR-AUC; recording time still unverified (open item)
- **Uncertainty:** candidate differences are within CV noise; the choice rests on calibration and is documented as such
- **Not causal, not production-ready, not a replacement for advisers:** the tool proposes a short list; a qualified adviser decides, may override, and records the decision locally
- Full text: `reports/limitations.md`, `reports/bias_fairness_analysis.md`, `reports/model_card.md`